# UEFA Euro 2020 Final — Visualizations
**Italy 1–1 England (AET) — Italy won 3–2 on penalties** | Wembley Stadium, London | 11 July 2021

Loads processed events from `cache/` and saves all figures to `figures/`.

Goals (regulation + extra time):
- 2' Luke Shaw (England)
- 67' Leonardo Bonucci (Italy)

**360 / freeze-frame data:** Available for UEFA Euro 2020 in StatsBomb open data. `shot_freeze_frame.png` shows player positions at the moment of each goal.

**Extra time:** This match went to extra time (periods 3 and 4 in StatsBomb data). xG timeline and momentum charts cover all 120 minutes.

## 0. Setup & Data Load

In [1]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as mpe
from matplotlib.lines import Line2D
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from scipy.ndimage import gaussian_filter
from mplsoccer import Pitch, VerticalPitch
warnings.filterwarnings('ignore')

BASE  = r'D:\Masaüstü\Projects\MatchAnalysis\Italy-England2020'
CACHE = os.path.join(BASE, 'cache')
FIGS  = os.path.join(BASE, 'figures')
os.makedirs(FIGS, exist_ok=True)

PITCH_COLOR = '#1a1a2e'
LINE_COLOR  = 'white'
ITA_COLOR   = '#009246'   # Italy green
ENG_COLOR   = '#003090'   # England blue
DPI = 150

plt.rcParams['font.family'] = 'DejaVu Sans'

def save(name):
    path = os.path.join(FIGS, name)
    plt.savefig(path, dpi=DPI, bbox_inches='tight', facecolor=plt.gcf().get_facecolor())
    plt.close('all')
    size = os.path.getsize(path) // 1024
    print(f'  saved {name}  ({size} KB)')

# Load
events = pd.read_pickle(os.path.join(CACHE, 'events_processed.pkl'))
with open(os.path.join(CACHE, 'meta.json'), encoding='utf-8') as f:
    meta = json.load(f)

ITA = meta['ITA']   # 'Italy'
ENG = meta['ENG']   # 'England'
has_360 = meta.get('has_360', False)

shots      = events[events['type'] == 'Shot'].copy()
passes     = events[events['type'] == 'Pass'].copy()
carries    = events[events['type'] == 'Carry'].copy()
presses    = events[events['type'] == 'Pressure'].copy()
blocks     = events[events['type'] == 'Block'].copy()
intercepts = events[events['type'] == 'Interception'].copy()
dribbles   = events[events['type'] == 'Dribble'].copy()
touches    = pd.concat([passes[['team','player','x','y']],
                        carries[['team','player','x','y']]])

# Goals in regulation + ET only (periods 1-4); period 5 = penalty shootout
goals_df = shots[(shots['shot_outcome'] == 'Goal') &
                 (shots['period'] <= 4)][['team','minute','period','player','x','y']].copy()

# Minute normalisation: period 3 => 90+, period 4 => 105+
def normalise_minute(row):
    p, m = row['period'], row['minute']
    if p == 1: return min(m, 45)
    if p == 2: return min(m, 90)
    if p == 3: return 90 + min(m - 90, 15)
    if p == 4: return 105 + min(m - 105, 15)
    return m

goals_plot = shots[(shots['shot_outcome'] == 'Goal') & (shots['period'] <= 4)].copy()
goals_plot['norm_min'] = goals_plot.apply(normalise_minute, axis=1)

all_players = events['player'].dropna().unique()
shaw_name    = next((p for p in all_players if 'Shaw'    in p), None)
bonucci_name = next((p for p in all_players if 'Bonucci' in p), None)

print(f'Events: {len(events)} | Shots: {len(shots)} | Passes: {len(passes)}')
print(f'has_360: {has_360}')
print(f'Goals (reg+ET):\n{goals_df[["player","team","minute","period"]].to_string(index=False)}')

Events: 4796 | Shots: 35 | Passes: 1350
has_360: True
Goals (reg+ET):
          player    team  minute  period
       Luke Shaw England       1       1
Leonardo Bonucci   Italy      66       2


## 1. Shot Map  `figures/shot_map.png`

In [2]:
reg_shots = shots[shots['period'] <= 4].copy()

pitch = VerticalPitch(pitch_type='statsbomb', pitch_color=PITCH_COLOR,
                      line_color=LINE_COLOR, half=True, line_zorder=2)
fig, ax = pitch.draw(figsize=(8, 7))
fig.set_facecolor(PITCH_COLOR)

for _, row in reg_shots.iterrows():
    color   = ITA_COLOR if row['team'] == ITA else ENG_COLOR
    xg      = row.get('shot_statsbomb_xg', 0.05)
    xg      = xg if pd.notna(xg) else 0.05
    outcome = str(row.get('shot_outcome', ''))
    marker  = '*' if outcome == 'Goal' else ('o' if outcome == 'Saved' else 'X')
    pitch.scatter(row['x'], row['y'], ax=ax,
                  s=xg * 1800 + 80, c=color, marker=marker,
                  edgecolors='white', linewidths=0.7, alpha=0.92, zorder=5)

legend_elements = [
    mpatches.Patch(color=ITA_COLOR, label='Italy'),
    mpatches.Patch(color=ENG_COLOR, label='England'),
    Line2D([0],[0], marker='*', color='w', label='Goal',          markersize=12, linewidth=0),
    Line2D([0],[0], marker='o', color='w', label='Saved',         markersize=8,  linewidth=0),
    Line2D([0],[0], marker='X', color='w', label='Off / Blocked', markersize=8,  linewidth=0),
]
ax.legend(handles=legend_elements, loc='lower left', facecolor='#0d0d1f',
          labelcolor='white', fontsize=8, framealpha=0.8)
ax.set_title('Shot Map — UEFA Euro 2020 Final\n'
             'Italy 1–1 England AET  (bubble size = xG)',
             color='white', fontsize=10, pad=10)
save('shot_map.png')

  saved shot_map.png  (66 KB)


## 2. Pass Networks  `figures/pass_network_italy.png` · `figures/pass_network_england.png`

In [3]:
def build_pass_network(team_name, team_color, fname):
    team_events = events[(events['team'] == team_name) & (events['period'].isin([1, 2]))]
    team_passes = team_events[team_events['type'] == 'Pass'].copy()
    completed   = team_passes[team_passes['pass_outcome'].isna() &
                              team_passes['pass_recipient'].notna()]

    avg_pos      = team_events[team_events['x'].notna()].groupby('player')[['x','y']].mean()
    event_counts = team_events[team_events['x'].notna()].groupby('player').size().rename('count')
    player_df    = avg_pos.join(event_counts).dropna()

    starters_row = events[(events['team'] == team_name) & (events['type'] == 'Starting XI')]
    top11 = []
    if len(starters_row):
        tactics = starters_row.iloc[0].get('tactics', {})
        if isinstance(tactics, dict) and 'lineup' in tactics:
            top11 = [p['player']['name'] for p in tactics['lineup']
                     if p['player']['name'] in player_df.index]
    if len(top11) < 8:
        top11 = player_df['count'].sort_values(ascending=False).head(11).index.tolist()

    player_df = player_df[player_df.index.isin(top11)]

    edges = {}
    for _, row in completed.iterrows():
        src = row['player']; dst = row['pass_recipient']
        if src in top11 and dst in top11 and src != dst:
            key = tuple(sorted([src, dst]))
            edges[key] = edges.get(key, 0) + 1

    pitch = Pitch(pitch_type='statsbomb', pitch_color=PITCH_COLOR,
                  line_color=LINE_COLOR, line_zorder=2)
    fig, ax = pitch.draw(figsize=(12, 8))
    fig.set_facecolor(PITCH_COLOR)

    max_edge  = max(edges.values(), default=1)
    max_count = player_df['count'].max() if len(player_df) else 1

    for (src, dst), cnt in edges.items():
        if src not in player_df.index or dst not in player_df.index:
            continue
        x1, y1 = player_df.loc[src, ['x','y']]
        x2, y2 = player_df.loc[dst, ['x','y']]
        lw = 0.5 + 5.5 * cnt / max_edge
        ax.plot([x1, x2], [y1, y2], color=team_color, alpha=0.5, lw=lw, zorder=3)

    for player, row in player_df.iterrows():
        size = 300 + 1200 * row['count'] / max_count
        ax.scatter(row['x'], row['y'], s=size, color=team_color,
                   edgecolors='white', linewidths=1.5, zorder=6)
        short = player.split()[-1]
        ax.text(row['x'], row['y'] + 3.5, short, color='white', fontsize=7,
                ha='center', va='bottom', zorder=7,
                path_effects=[mpe.withStroke(linewidth=2, foreground=PITCH_COLOR)])

    ax.set_title(f'{team_name} Pass Network — UEFA Euro 2020 Final\n'
                 '(node size = total actions, edge width = completed pass pairs)',
                 color='white', fontsize=11, pad=10)
    save(fname)

build_pass_network(ITA, ITA_COLOR, 'pass_network_italy.png')
build_pass_network(ENG, ENG_COLOR, 'pass_network_england.png')

  saved pass_network_italy.png  (206 KB)


  saved pass_network_england.png  (204 KB)


## 3. Touch Heatmaps  `figures/heatmap_italy.png` · `figures/heatmap_england.png`

In [4]:
def team_heatmap(team_name, fname):
    team_touches = touches[touches['team'] == team_name].dropna(subset=['x','y'])
    x = team_touches['x'].values.astype(float)
    y = team_touches['y'].values.astype(float)

    pitch = Pitch(pitch_type='statsbomb', pitch_color=PITCH_COLOR,
                  line_color=LINE_COLOR, line_zorder=2)
    fig, ax = pitch.draw(figsize=(12, 8))
    fig.set_facecolor(PITCH_COLOR)

    pitch.kdeplot(x, y, ax=ax, cmap='hot', fill=True, alpha=0.65,
                  levels=50, thresh=0.05, zorder=3)
    ax.set_title(f'{team_name} — Touch Heatmap (KDE)\nUEFA Euro 2020 Final',
                 color='white', fontsize=11, pad=10)
    save(fname)

team_heatmap(ITA, 'heatmap_italy.png')
team_heatmap(ENG, 'heatmap_england.png')

  saved heatmap_italy.png  (184 KB)


  saved heatmap_england.png  (191 KB)


## 4. Defensive Actions  `figures/defensive_actions.png`

In [5]:
def_events = pd.concat([presses, blocks, intercepts]).dropna(subset=['x','y'])

pitch = Pitch(pitch_type='statsbomb', pitch_color=PITCH_COLOR,
              line_color=LINE_COLOR, line_zorder=2)
fig, axes = pitch.draw(figsize=(14, 8), ncols=2)
fig.set_facecolor(PITCH_COLOR)
fig.suptitle('Defensive Actions — Pressures, Blocks & Interceptions\nUEFA Euro 2020 Final',
             color='white', fontsize=12, y=1.01)

type_cfg = {'Pressure': ('o', 0.35, 40), 'Block': ('s', 0.8, 80), 'Interception': ('^', 0.95, 110)}

for ax, (team, color) in zip(axes, [(ITA, ITA_COLOR), (ENG, ENG_COLOR)]):
    team_def = def_events[def_events['team'] == team]
    for etype, (marker, alpha, size) in type_cfg.items():
        sub = team_def[team_def['type'] == etype]
        if len(sub):
            pitch.scatter(sub['x'], sub['y'], ax=ax, s=size, c=color,
                          marker=marker, alpha=alpha, edgecolors='white',
                          linewidths=0.4, zorder=5)
    ax.set_title(f'{team}  (n={len(team_def)})', color='white', fontsize=10, pad=8)

legend_elements = [
    Line2D([0],[0], marker='o', color='gray', label='Pressure',  markersize=7, linewidth=0),
    Line2D([0],[0], marker='s', color='gray', label='Block',     markersize=7, linewidth=0),
    Line2D([0],[0], marker='^', color='gray', label='Intercept', markersize=7, linewidth=0),
]
axes[1].legend(handles=legend_elements, loc='lower right', facecolor='#0d0d1f',
               labelcolor='white', fontsize=8, framealpha=0.8)
save('defensive_actions.png')

  saved defensive_actions.png  (155 KB)


## 5. xG Timeline (reg + extra time)  `figures/xg_timeline.png`

In [6]:
xg_shots = shots[(shots['period'] <= 4) & shots['shot_statsbomb_xg'].notna()].copy()
xg_shots['norm_min'] = xg_shots.apply(normalise_minute, axis=1)
xg_shots = xg_shots.sort_values('norm_min')

def cum_xg_df(team):
    df = xg_shots[xg_shots['team'] == team].copy().sort_values('norm_min')
    df['cum_xg'] = df['shot_statsbomb_xg'].cumsum()
    return df

ita_xg = cum_xg_df(ITA)
eng_xg = cum_xg_df(ENG)

fig, ax = plt.subplots(figsize=(13, 6))
fig.set_facecolor('#0d0d1f'); ax.set_facecolor('#0d0d1f')

def step_plot(df, color, label, end_min=120):
    last_val = df['cum_xg'].iloc[-1] if len(df) else 0
    xs = [0] + df['norm_min'].tolist() + [end_min]
    ys = [0] + df['cum_xg'].tolist()  + [last_val]
    ax.step(xs, ys, where='post', color=color, lw=2.5, label=label)

step_plot(ita_xg, ITA_COLOR, f'Italy   (xG={ita_xg["shot_statsbomb_xg"].sum():.2f})')
step_plot(eng_xg, ENG_COLOR, f'England (xG={eng_xg["shot_statsbomb_xg"].sum():.2f})')

goal_lookups = {ITA: ita_xg, ENG: eng_xg}
goal_colors  = {ITA: ITA_COLOR, ENG: ENG_COLOR}

for _, g in goals_plot.iterrows():
    team = g['team']; nm = g['norm_min']
    cum_df = goal_lookups[team]
    before = cum_df[cum_df['norm_min'] <= nm]
    cum_val = before['cum_xg'].max() if len(before) else 0
    ax.axvline(nm, color=goal_colors[team], linestyle='--', alpha=0.55, lw=1.2)
    ax.scatter(nm, cum_val, color=goal_colors[team], s=130, zorder=6,
               edgecolors='white', linewidths=1.2)
    ax.text(nm + 0.7, cum_val + 0.015,
            f"{int(g['minute'])}' {g['player'].split()[-1]}",
            color='white', fontsize=8, va='bottom')

ax.axvline(45,  color='gray', linestyle=':', alpha=0.5, lw=1)
ax.axvline(90,  color='gray', linestyle=':', alpha=0.5, lw=1)
ax.axvline(105, color='gray', linestyle=':', alpha=0.3, lw=0.8)
ax.text(45.5,  0.01, 'HT',    color='gray', fontsize=8)
ax.text(90.5,  0.01, 'FT/ET', color='gray', fontsize=8)
ax.text(105.5, 0.01, 'ET HT', color='gray', fontsize=7)

ax.set_xlim(0, 121)
ax.set_xlabel('Minute (incl. Extra Time)', color='white')
ax.set_ylabel('Cumulative xG', color='white')
ax.tick_params(colors='white'); ax.spines[:].set_edgecolor('#555')
ax.set_title('Cumulative xG Timeline — UEFA Euro 2020 Final\n'
             'Italy 1–1 England AET (Italy won 3–2 on penalties)',
             color='white', fontsize=12, pad=10)
ax.legend(facecolor='#1a1a2e', labelcolor='white', fontsize=9)
ax.grid(axis='y', color='#333', linewidth=0.5)
save('xg_timeline.png')

  saved xg_timeline.png  (75 KB)


## 6. Match Momentum (120 min)  `figures/momentum.png`

In [7]:
WINDOW = 5
ev_mom = events[events['period'] <= 4].copy()
ev_mom['norm_min'] = ev_mom.apply(normalise_minute, axis=1).astype(float)

minutes = np.arange(1, 121)

def rolling_events(team, window=WINDOW):
    te = ev_mom[ev_mom['team'] == team]
    return np.array([
        te[(te['norm_min'] > m - window) & (te['norm_min'] <= m)].shape[0]
        for m in minutes])

ita_roll = rolling_events(ITA)
eng_roll = rolling_events(ENG)
diff     = ita_roll - eng_roll

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 9), sharex=True)
fig.set_facecolor('#0d0d1f')
for ax in [ax1, ax2]:
    ax.set_facecolor('#0d0d1f'); ax.tick_params(colors='white')
    ax.spines[:].set_edgecolor('#444'); ax.grid(axis='y', color='#333', lw=0.5)

ax1.fill_between(minutes, ita_roll, alpha=0.30, color=ITA_COLOR)
ax1.plot(minutes, ita_roll, color=ITA_COLOR, lw=1.8, label='Italy')
ax1.fill_between(minutes, eng_roll, alpha=0.30, color=ENG_COLOR)
ax1.plot(minutes, eng_roll, color=ENG_COLOR, lw=1.8, label='England')
ax1.set_ylabel('Events (5-min rolling)', color='white')
ax1.legend(facecolor='#1a1a2e', labelcolor='white', fontsize=9)
ax1.set_title('Match Momentum — UEFA Euro 2020 Final  '
              '(Italy 1–1 England AET · Italy won on penalties)',
              color='white', fontsize=12, pad=8)

colors_bar = [ITA_COLOR if d >= 0 else ENG_COLOR for d in diff]
ax2.bar(list(minutes), diff, color=colors_bar, alpha=0.75)
ax2.axhline(0, color='white', lw=0.8)
ax2.set_ylabel('Italy − England momentum', color='white')
ax2.set_xlabel('Minute (incl. Extra Time)', color='white')

for ax in [ax1, ax2]:
    ax.axvline(45,  color='gray', linestyle=':', lw=1, alpha=0.6)
    ax.axvline(90,  color='gray', linestyle=':', lw=1, alpha=0.6)
    ax.axvline(105, color='gray', linestyle=':', lw=0.7, alpha=0.4)
    for _, g in goals_plot.iterrows():
        c = ITA_COLOR if g['team'] == ITA else ENG_COLOR
        ax.axvline(g['norm_min'], color=c, linestyle='--', lw=1, alpha=0.7)

y_top = ax1.get_ylim()[1]
ax1.text(45.5,  y_top * 0.93, 'HT',    color='gray', fontsize=8)
ax1.text(90.5,  y_top * 0.93, 'FT/ET', color='gray', fontsize=8)
for _, g in goals_plot.iterrows():
    c = ITA_COLOR if g['team'] == ITA else ENG_COLOR
    ax1.text(g['norm_min'] + 0.6, y_top * 0.80,
             f"{int(g['minute'])}' {g['player'].split()[-1]}",
             color=c, fontsize=7.5, rotation=40, va='bottom')

plt.tight_layout()
save('momentum.png')

  saved momentum.png  (207 KB)


## 7. Player Radar Charts

Italy key players selected by event volume among midfielders/attackers, with goal scorer Bonucci guaranteed:
- Leonardo Bonucci (67' equaliser), Marco Verratti (top mid by event count), Jorginho (deep-lying pivot, top passer)

England:
- Luke Shaw (2' goal scorer), Kalvin Phillips (second most events), Harry Kane (striker)

In [8]:
METRICS = ['Passes', 'Carries', 'Shots', 'Pressures', 'Dribbles', 'Ball Recoveries', 'Duels']

def player_metrics(player):
    pe = events[events['player'] == player]
    return [
        len(pe[pe['type'] == 'Pass']),
        len(pe[pe['type'] == 'Carry']),
        len(pe[pe['type'] == 'Shot']),
        len(pe[pe['type'] == 'Pressure']),
        len(pe[pe['type'] == 'Dribble']),
        len(pe[pe['type'] == 'Ball Recovery']),
        len(pe[pe['type'] == 'Duel']),
    ]

def radar_chart(players, colors, team_label, fname):
    N      = len(METRICS)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False)
    angles_closed = np.append(angles, angles[0])

    all_vals = np.array([player_metrics(p) for p in players], dtype=float)
    col_max  = all_vals.max(axis=0)
    col_max[col_max == 0] = 1
    all_norm = all_vals / col_max

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    fig.set_facecolor('#0d0d1f'); ax.set_facecolor('#0d0d1f')

    ax.set_xticks(angles)
    ax.set_xticklabels(METRICS, color='white', fontsize=9)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(['25%','50%','75%','100%'], color='#999', fontsize=7)
    ax.tick_params(axis='x', pad=8)
    ax.spines['polar'].set_color('#444')
    for yt in [0.25, 0.5, 0.75, 1.0]:
        ax.plot(angles_closed, [yt]*len(angles_closed), '--', color='#444', lw=0.5, zorder=0)

    short_names = []
    for i, (player, color) in enumerate(zip(players, colors)):
        vals = np.append(all_norm[i], all_norm[i][0])
        ax.plot(angles_closed, vals, color=color, lw=2.2, zorder=4+i)
        ax.fill(angles_closed, vals, color=color, alpha=0.20, zorder=3+i)
        short_names.append(player.split()[-1])

    legend_elements = [mpatches.Patch(color=c, label=n) for c, n in zip(colors, short_names)]
    ax.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(1.38, 1.15),
              facecolor='#1a1a2e', labelcolor='white', fontsize=9)
    ax.set_title(f'{team_label} — Key Players Radar\nUEFA Euro 2020 Final',
                 color='white', fontsize=11, pad=25)
    save(fname)

# Italy: Bonucci (goal scorer) + top mids by event count, excluding defenders/GK
defenders_ita = {'Bonucci', 'Chiellini', 'Di Lorenzo', 'Emerson', 'Donnarumma', 'Palmieri'}
ita_ecounts   = events[events['team']==ITA].groupby('player').size().sort_values(ascending=False)
ita_non_def   = [p for p in ita_ecounts.index if not any(d in p for d in defenders_ita)]
ita_radar = [bonucci_name] if bonucci_name else []
for p in ita_non_def:
    if p not in ita_radar: ita_radar.append(p)
    if len(ita_radar) == 3: break

# England: Shaw (goal scorer) + top by event count
def_eng     = {'Pickford', 'Walker', 'Stones', 'Maguire', 'Trippier'}
eng_ecounts = events[events['team']==ENG].groupby('player').size().sort_values(ascending=False)
eng_non_def = [p for p in eng_ecounts.index if not any(d in p for d in def_eng)]
eng_radar = [shaw_name] if shaw_name else []
for p in eng_non_def:
    if p not in eng_radar: eng_radar.append(p)
    if len(eng_radar) == 3: break

print('Italy radar:', ita_radar)
print('England radar:', eng_radar)

radar_chart(ita_radar, ['#e8d44d', '#ff6b6b', '#ff9f43'], 'Italy',   'radar_italy_key_players.png')
radar_chart(eng_radar, ['#74b9ff', '#fd79a8', '#55efc4'], 'England', 'radar_england_key_players.png')

Italy radar: ['Leonardo Bonucci', 'Marco Verratti', 'Jorge Luiz Frello Filho']
England radar: ['Luke Shaw', 'Kalvin Phillips', 'Harry Kane']


  saved radar_italy_key_players.png  (253 KB)


  saved radar_england_key_players.png  (269 KB)


## 8. Team Stats Comparison  `figures/team_stats_comparison.png`

In [9]:
def team_summary(team):
    te  = events[(events['team'] == team) & (events['period'] <= 4)]
    p   = te[te['type'] == 'Pass']
    s   = te[te['type'] == 'Shot']
    sot = s[s['shot_outcome'].isin(['Saved','Goal'])]
    pr  = te[te['type'] == 'Pressure']
    comp_passes = p[p['pass_outcome'].isna()]
    pass_acc    = 100 * len(comp_passes) / max(len(p), 1)
    xg_total    = s['shot_statsbomb_xg'].sum()
    return {
        'Shots':           len(s),
        'Shots on Target': len(sot),
        'xG':              round(xg_total, 2),
        'Passes':          len(p),
        'Pass Acc (%)':    round(pass_acc, 1),
        'Pressures':       len(pr),
    }

ita_stats = team_summary(ITA)
eng_stats = team_summary(ENG)
print(pd.DataFrame([ita_stats, eng_stats], index=[ITA, ENG]).T.to_string())

metrics = list(ita_stats.keys())
x_pos   = np.arange(len(metrics))
width   = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
fig.set_facecolor('#0d0d1f'); ax.set_facecolor('#0d0d1f')

bars1 = ax.bar(x_pos - width/2, [ita_stats[m] for m in metrics],
               width, label='Italy', color=ITA_COLOR, alpha=0.85)
bars2 = ax.bar(x_pos + width/2, [eng_stats[m] for m in metrics],
               width, label='England', color=ENG_COLOR, alpha=0.85)

for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    lbl = f'{h:.2f}' if h < 5 else (f'{h:.1f}' if h < 20 else f'{int(h)}')
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.4, lbl,
            ha='center', va='bottom', color='white', fontsize=8)

ax.set_xticks(x_pos)
ax.set_xticklabels(metrics, color='white', fontsize=10)
ax.tick_params(colors='white'); ax.spines[:].set_edgecolor('#444')
ax.grid(axis='y', color='#333', lw=0.5)
ax.set_title('Team Statistics Comparison — UEFA Euro 2020 Final\n'
             'Italy 1–1 England AET (Italy won 3–2 on penalties)',
             color='white', fontsize=12, pad=10)
ax.legend(facecolor='#1a1a2e', labelcolor='white', fontsize=10)
save('team_stats_comparison.png')

                  Italy  England
Shots             19.00     6.00
Shots on Target    5.00     1.00
xG                 2.08     0.43
Passes           885.00   465.00
Pass Acc (%)      88.20    75.50
Pressures        157.00   213.00


  saved team_stats_comparison.png  (57 KB)


## 9. Shot Freeze-Frame (360 data)  `figures/shot_freeze_frame.png`

360-degree data is available for UEFA Euro 2020. This figure shows all player positions
at the moment of each goal — Luke Shaw (2') and Leonardo Bonucci (67').

In [10]:
if has_360:
    try:
        mid = meta['match_id']
        frames_raw = pd.read_json(os.path.join(CACHE, 'frames_' + str(mid) + '.json'))
        goal_events = shots[(shots['shot_outcome'] == 'Goal') &
                            (shots['period'] <= 4)].copy()

        n_goals = len(goal_events)
        fig, axes = plt.subplots(1, n_goals, figsize=(8 * n_goals, 7))
        fig.set_facecolor('#0d0d1f')
        if n_goals == 1:
            axes = [axes]
        fig.suptitle('Freeze-Frame: Player Positions at Goal Moments | UEFA Euro 2020 Final',
                     color='white', fontsize=12, y=1.02)

        pitch_ff = Pitch(pitch_type='statsbomb', pitch_color=PITCH_COLOR,
                         line_color=LINE_COLOR, line_zorder=2)

        for ax, (_, goal) in zip(axes, goal_events.iterrows()):
            pitch_ff.draw(ax=ax)
            ev_id = goal.get('id', None)
            player_rows = frames_raw[frames_raw['id'] == ev_id] if ev_id is not None else pd.DataFrame()

            for _, prow in player_rows.iterrows():
                loc = prow.get('location', None)
                if not isinstance(loc, list) or len(loc) < 2:
                    continue
                px, py = loc[0], loc[1]
                teammate = bool(prow.get('teammate', False))
                gk       = bool(prow.get('keeper', False))
                sc = ITA_COLOR if goal['team'] == ITA else ENG_COLOR
                oc = ENG_COLOR if goal['team'] == ITA else ITA_COLOR
                color  = sc if teammate else oc
                marker = 'D' if gk else 'o'
                ax.scatter(px, py, s=140, c=color, marker=marker,
                           edgecolors='white', linewidths=1.2, zorder=5, alpha=0.9)

            ax.scatter(goal['x'], goal['y'], s=400, c='yellow', marker='*',
                       edgecolors='white', linewidths=1.5, zorder=7)
            tl  = 'Italy' if goal['team'] == ITA else 'England'
            cl  = ITA_COLOR if goal['team'] == ITA else ENG_COLOR
            nm  = goal['player'].split()[-1]
            mn  = str(int(goal['minute']))
            ttl = nm + '  ' + mn + 'min  (' + tl + ')  [' + str(len(player_rows)) + ' tracked]'
            ax.set_title(ttl, color=cl, fontsize=10, fontweight='bold', pad=8)

        legend_el = [
            mpatches.Patch(color=ITA_COLOR, label='Italy'),
            mpatches.Patch(color=ENG_COLOR, label='England'),
            Line2D([0],[0], marker='D', color='w', label='Goalkeeper', markersize=8, linewidth=0),
            Line2D([0],[0], marker='o', color='w', label='Outfield',   markersize=8, linewidth=0),
            Line2D([0],[0], marker='*', color='yellow', label='Shot',  markersize=12, linewidth=0),
        ]
        axes[-1].legend(handles=legend_el, loc='lower right', facecolor='#0d0d1f',
                        labelcolor='white', fontsize=8, framealpha=0.8)
        plt.tight_layout()
        save('shot_freeze_frame.png')
    except Exception as e:
        print('Freeze-frame skipped:', str(e))
        plt.close('all')
else:
    print('360 data not available.')


  saved shot_freeze_frame.png  (101 KB)


## 10. Pressure Map  `figures/pressure_map.png`

In [11]:
def kde_overlay(ax, x_vals, y_vals, cmap='hot', alpha=0.65):
    from scipy.stats import gaussian_kde
    if len(x_vals) < 5: return
    xy  = np.vstack([x_vals, y_vals])
    kde = gaussian_kde(xy, bw_method=0.25)
    xi  = np.linspace(0, 120, 250); yi = np.linspace(0, 80, 250)
    Xi, Yi = np.meshgrid(xi, yi)
    Zi = gaussian_filter(
        kde(np.vstack([Xi.ravel(), Yi.ravel()])).reshape(Xi.shape), sigma=2)
    Zi = (Zi - Zi.min()) / (Zi.max() - Zi.min() + 1e-12)
    ax.contourf(Xi, Yi, Zi, levels=25, cmap=cmap, alpha=alpha, zorder=3)

pitch = Pitch(pitch_type='statsbomb', pitch_color=PITCH_COLOR,
              line_color=LINE_COLOR, line_zorder=2)
fig, axes = pitch.draw(figsize=(14, 8), ncols=2)
fig.set_facecolor(PITCH_COLOR)
fig.suptitle('Pressure Map — UEFA Euro 2020 Final',
             color='white', fontsize=13, y=1.01)

for ax, (team, color) in zip(axes, [(ITA, ITA_COLOR), (ENG, ENG_COLOR)]):
    team_press = presses[(presses['team'] == team)].dropna(subset=['x','y'])
    if len(team_press) >= 5:
        kde_overlay(ax, team_press['x'].values, team_press['y'].values)
    ax.set_title(f'{team}  ({len(team_press)} pressures)', color='white', fontsize=10, pad=8)

save('pressure_map.png')

  saved pressure_map.png  (183 KB)


## Turkish-Named Extended Set (gorsel_01 – gorsel_14)

The script `_run_viz.py` produces all 14 Turkish-named figures. These are generated separately
for speed; re-run that script if you need to regenerate them.

| File | Description |
|---|---|
| gorsel_01_oyuncu_isi.png | Player touch heatmap (top player per team) |
| gorsel_02_pas_isi.png | Team pass density heatmap |
| gorsel_03_sut.png | Shot map (half pitch) |
| gorsel_04_pas_agi.png | Pass network (1st half) |
| gorsel_05_xg.png | xG timeline |
| gorsel_06_savunma.png | Defensive actions |
| gorsel_07_ilerletici_pas.png | Progressive pass map (≥10m) |
| gorsel_08_radar.png | Verratti vs Rice · Bonucci vs Mount |
| gorsel_09_momentum.png | Momentum (120 min) |
| gorsel_10_bolge.png | Zone control (6-zone split) |
| gorsel_11_dribbling.png | Dribble map |
| gorsel_12_pas_yonu.png | Pass direction rose |
| gorsel_13_counter_press.png | Counter-press heatmap |
| gorsel_14_gol_yapilanma.png | Goal buildup map (last 8 events) |

## Summary

In [12]:
print(f'Figures in {FIGS}:')
for fn in sorted(os.listdir(FIGS)):
    size = os.path.getsize(os.path.join(FIGS, fn)) // 1024
    print(f'  {fn:50s}  {size:5d} KB')
print()
ita_xg_total = shots[(shots['team']==ITA)&(shots['period']<=4)]['shot_statsbomb_xg'].sum()
eng_xg_total = shots[(shots['team']==ENG)&(shots['period']<=4)]['shot_statsbomb_xg'].sum()
print(f'Italy   xG: {ita_xg_total:.3f}')
print(f'England xG: {eng_xg_total:.3f}')

Figures in D:\Masaüstü\Projects\MatchAnalysis\Italy-England2020\figures:
  defensive_actions.png                                 155 KB
  gorsel_01_oyuncu_isi.png                              179 KB
  gorsel_02_pas_isi.png                                 249 KB
  gorsel_03_sut.png                                      68 KB
  gorsel_04_pas_agi.png                                 234 KB
  gorsel_05_xg.png                                       70 KB
  gorsel_06_savunma.png                                 200 KB
  gorsel_07_ilerletici_pas.png                          709 KB
  gorsel_08_radar.png                                   363 KB
  gorsel_09_momentum.png                                205 KB
  gorsel_10_bolge.png                                    78 KB
  gorsel_11_dribbling.png                               111 KB
  gorsel_12_pas_yonu.png                                490 KB
  gorsel_13_counter_press.png                           304 KB
  gorsel_14_gol_yapilanma.png                